# Thali — multi-task SmolVLA fine-tune (Kaggle, 2× T4 or P100)

Fine-tunes `lerobot/smolvla_base` on `Prashant-77/thali_all` (1050 scripted-expert episodes (742 837 frames), 7 skills, 3 cameras 240×320, 12-D actions, 10 paraphrases per skill) and pushes the checkpoint to `Prashant-77/thali_smolvla`.

**Before running:** add a Kaggle secret `HF_TOKEN` (write access to Prashant-77). Enable GPU (T4 ×2 or P100) and Internet in the notebook settings. Expected wall time ≈ 5–6 h for 20 000 steps at batch 16.

In [ ]:
!pip -q install 'lerobot==0.4.4' 'huggingface_hub>=0.30'
import torch, lerobot; print(torch.__version__, torch.cuda.get_device_name(0), lerobot.__version__)

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
from huggingface_hub import login, whoami
login(token=os.environ['HF_TOKEN']); print(whoami()['name'])

In [ ]:
# dataset: streamed from the Hub into the LeRobot cache
from lerobot.datasets.lerobot_dataset import LeRobotDataset
ds = LeRobotDataset('Prashant-77/thali_all')
print(ds.num_episodes, 'episodes', ds.num_frames, 'frames', ds.fps, 'fps'); print(ds.meta.features.keys()); print(ds.meta.tasks.head())

In [ ]:
# Training. SmolVLA config follows the plan (plan §3 Phase 3.1) with a batch that fits a T4 (16 GB): 16 x 3 cameras at 240x320
# (resized with padding to 512 by the policy), AMP on, vision encoder frozen (default), action expert + state projection trained.
!lerobot-train \
  --policy.path=lerobot/smolvla_base --policy.device=cuda --policy.use_amp=true \
  --dataset.repo_id=Prashant-77/thali_all \
  --batch_size=16 --steps=20000 --log_freq=200 --save_freq=5000 --eval_freq=0 --num_workers=4 \
  --policy.chunk_size=50 --policy.n_action_steps=50 \
  --output_dir=/kaggle/working/outputs/smolvla_mt --job_name=thali_smolvla --wandb.enable=false \
  --policy.push_to_hub=true --policy.repo_id=Prashant-77/thali_smolvla --seed=1000

In [ ]:
# Push the final checkpoint (in case push_to_hub was interrupted) and a model card.
from huggingface_hub import HfApi
api = HfApi()
api.create_repo('Prashant-77/thali_smolvla', exist_ok=True)
api.upload_folder(folder_path='/kaggle/working/outputs/smolvla_mt/checkpoints/last/pretrained_model', repo_id='Prashant-77/thali_smolvla')
api.upload_file(path_or_fileobj=b'''---
license: apache-2.0
base_model: lerobot/smolvla_base
datasets: [Prashant-77/thali_all]
tags: [lerobot, smolvla, so101, bimanual, mujoco, thali]
---
# Thali multi-task SmolVLA
Fine-tuned on 1050 scripted-expert episodes of the Thali dinner-table task (7 skills, language-conditioned).
Evaluated back in the MuJoCo env by `eval/run_seeds.py --policy smolvla` in the Thali repo.
''', path_in_repo='README.md', repo_id='Prashant-77/thali_smolvla')
print('pushed Prashant-77/thali_smolvla')

## After it finishes
Back on the laptop: `make eval POLICY=smolvla` pulls `Prashant-77/thali_smolvla` and adds the SmolVLA rows to `results/seeds.json`. Until then those rows read "pending SmolVLA run".